# Linear Regression — Student_Performance.csv
Predict **Performance Index** from Hours Studied, Previous Scores, Extracurricular Activities, Sleep Hours, Sample Question Papers Practiced.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("Student_Performance.csv")
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91
1,4,82,No,4,2,65
2,8,51,Yes,7,2,45
3,5,52,Yes,5,2,36
4,7,75,No,8,5,66


In [2]:
df.shape

(10000, 6)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   Hours Studied                     10000 non-null  int64
 1   Previous Scores                   10000 non-null  int64
 2   Extracurricular Activities        10000 non-null  str  
 3   Sleep Hours                       10000 non-null  int64
 4   Sample Question Papers Practiced  10000 non-null  int64
 5   Performance Index                 10000 non-null  int64
dtypes: int64(5), str(1)
memory usage: 468.9 KB


## 2. Preprocessing

In [4]:
print("Missing values:\n", df.isna().sum())
df = df.dropna()

Missing values:
 Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64


In [5]:
# Encode categorical column: Extracurricular Activities (Yes/No)
categorical_cols = ["Extracurricular Activities"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.head()

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index,Extracurricular Activities_Yes
0,7,99,9,1,91,True
1,4,82,4,2,65,False
2,8,51,7,2,45,True
3,5,52,5,2,36,True
4,7,75,8,5,66,False


In [6]:
target_col = "Performance Index"
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]
feature_columns = X.columns.tolist()
feature_columns

['Hours Studied',
 'Previous Scores',
 'Sleep Hours',
 'Sample Question Papers Practiced',
 'Extracurricular Activities_Yes']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

Train shape: (8000, 5)  Test shape: (2000, 5)


## 3. Model Training

In [8]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Coefficients:", dict(zip(X.columns, model.coef_)))
print("Intercept:", model.intercept_)

Coefficients: {'Hours Studied': np.float64(2.8524839300725775), 'Previous Scores': np.float64(1.016988198932932), 'Sleep Hours': np.float64(0.476941484176272), 'Sample Question Papers Practiced': np.float64(0.19183144145054234), 'Extracurricular Activities_Yes': np.float64(0.6086166795764197)}
Intercept: -33.92194621555638


## 4. Model Evaluation — MAE & R² Score

In [9]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Absolute Error (MAE): 1.6111
R2 Score: 0.9890


In [10]:
pd.DataFrame({"Actual": y_test, "Predicted": y_pred}).head(10)

,Actual,Predicted
6252,51,54.711854
4684,20,22.615513
1731,46,47.903145
4742,28,31.289767
4521,41,43.004570
6340,59,59.071252
576,48,45.903475
5202,87,86.459118
6363,37,37.700140
439,73,72.055925


## 5. Predictive Function

In [11]:
def predict_performance(hours_studied, previous_scores,
                         extracurricular_activities, sleep_hours,
                         sample_question_papers_practiced):
    """
    Predict a student's Performance Index.
    extracurricular_activities: 'Yes' or 'No'
    """
    raw_input = pd.DataFrame([{
        "Hours Studied": hours_studied,
        "Previous Scores": previous_scores,
        "Extracurricular Activities": extracurricular_activities,
        "Sleep Hours": sleep_hours,
        "Sample Question Papers Practiced": sample_question_papers_practiced
    }])
    raw_input = pd.get_dummies(raw_input, columns=categorical_cols, drop_first=True)
    raw_input = raw_input.reindex(columns=feature_columns, fill_value=0)
    prediction = model.predict(raw_input)[0]
    return prediction

## 6. Prediction (example)

In [12]:
predicted_score = predict_performance(
    hours_studied=6, previous_scores=80, extracurricular_activities="Yes",
    sleep_hours=7, sample_question_papers_practiced=4
)
print(f"Predicted Performance Index: {predicted_score:.2f}")

Predicted Performance Index: 68.66
